In [ ]:
# # N 建议先取 20；若 top-2 答案频率接近，再增大到 50~100
# N = 20
# answers = []
# for i in range(N):
#     resp = llm(prompt, temperature=0.7)          # 多次独立采样
#     answers.append(extract_final_answer(resp))    # 答案抽取是关键环节

# final = mode(answers)                             # 多数投票
# confidence = answers.count(final) / N             # 出现频率 ≈ 置信度代理

# # 建议同时报告 top-2 的频率差，用于判断是否需要人工介入
# top2 = Counter(answers).most_common(2)
# margin = top2[0][1] / N - (top2[1][1] / N if len(top2) > 1 else 0.0)

In [1]:
import numpy as np

def wilson(x, n, z=1.959963985):
    ph = x / n
    den = 1 + z * z / n
    center = (ph + z * z / (2 * n)) / den
    half = z * np.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / den
    return center - half, center + half

def vote_accuracy(p_correct, k, n_opts, n_trials, rng):
    # 每次作答：以 p_correct 答对；答错时在 n_opts-1 个干扰项上均匀分布
    correct = rng.random((n_trials, k)) < p_correct
    distractor = rng.integers(0, n_opts - 1, size=(n_trials, k))
    counts = np.zeros((n_trials, n_opts), dtype=np.int64)
    counts[:, 0] = correct.sum(axis=1)
    for d in range(n_opts - 1):
        counts[:, d + 1] = ((~correct) & (distractor == d)).sum(axis=1)
    # 得票最高者获胜；平票时等概率随机决出
    m = counts.max(axis=1, keepdims=True)
    is_top = counts == m
    win_prob = np.where(is_top[:, 0], 1.0 / is_top.sum(axis=1), 0.0)
    wins = rng.random(n_trials) < win_prob
    est = wins.mean()
    lo, hi = wilson(int(wins.sum()), n_trials)
    return est, lo, hi

rng = np.random.default_rng(20240)
n_trials = 40000

print("=== 设定一：二选一（Condorcet 条件直观成立） ===")
print("k     p=0.55（投票受益）   p=0.45（投票受害）")
for k in [1, 3, 5, 10, 20, 40]:
    est1, _, _ = vote_accuracy(0.55, k, 2, n_trials, rng)
    est2, _, _ = vote_accuracy(0.45, k, 2, n_trials, rng)
    print(f"{k:<5d} {est1:.4f}              {est2:.4f}")

print()
print("=== 设定二：四选一（干扰项分摊错误票） ===")
print("k     p=0.40              p=0.55")
for k in [1, 3, 5, 10, 20, 40]:
    est1, _, _ = vote_accuracy(0.40, k, 4, n_trials, rng)
    est2, _, _ = vote_accuracy(0.55, k, 4, n_trials, rng)
    print(f"{k:<5d} {est1:.4f}              {est2:.4f}")

print()
print("=== 设定二（p=0.55）的 Wilson 95% 区间（n_trials=40000） ===")
for k in [10, 20]:
    est, lo, hi = vote_accuracy(0.55, k, 4, n_trials, rng)
    print(f"k={k:<3d} 投票正确率 {est:.4f}  95%CI [{lo:.4f}, {hi:.4f}]")

=== 设定一：二选一（Condorcet 条件直观成立） ===
k     p=0.55（投票受益）   p=0.45（投票受害）
1     0.5462              0.4500
3     0.5710              0.4261
5     0.5962              0.4021
10    0.6188              0.3775
20    0.6734              0.3323
40    0.7347              0.2614

=== 设定二：四选一（干扰项分摊错误票） ===
k     p=0.40              p=0.55
1     0.4031              0.5527
3     0.4477              0.6460
5     0.5099              0.7454
10    0.6157              0.8835
20    0.7494              0.9752
40    0.8901              0.9984

=== 设定二（p=0.55）的 Wilson 95% 区间（n_trials=40000） ===
k=10  投票正确率 0.8830  95%CI [0.8798, 0.8861]
k=20  投票正确率 0.9737  95%CI [0.9721, 0.9752]


In [ ]:
# import dspy

# lm = dspy.LM("openai/gpt-4o-mini", temperature=0.0)
# dspy.configure(lm=lm)

# # 1) 只声明"输入→输出"的签名，不手写提示词
# class MethodChoice(dspy.Signature):
#     """为给定的数据分析场景推荐统计检验方法。"""
#     scenario: str = dspy.InputField()
#     method: str = dspy.OutputField()          # 推荐的方法名
#     assumptions: str = dspy.OutputField()     # 必须显式列出的假设

# # 2) 定义评价指标（这里用可程序化判定的弱指标示意）
# def metric(example, pred, trace=None) -> float:
#     ok_method = example.method.lower() in pred.method.lower()
#     ok_assume = len(pred.assumptions) > 20
#     return float(ok_method and ok_assume)

# # 3) 用小验证集自动编译（自动搜索示例与指令）
# optimizer = dspy.MIPROv2(metric=metric, auto="light")
# program = optimizer.compile(dspy.Predict(MethodChoice), trainset=train_set)

# print(program(scenario="两组各 50 例，连续终点，8% 缺失").method)

In [3]:
import numpy as np

rng = np.random.default_rng(20241)
B = 24          # 题目数（区组数）
n_sim = 20000   # 模拟重复次数
delta = 0.5     # 变体 B 相对 A 的真实效应（logit 尺度）
sigma_b = 1.5   # 题目难度（区组效应）的组间标准差

a = rng.normal(0, sigma_b, size=B)                       # 固定的题目难度
p0 = 1 / (1 + np.exp(-(a + 0.0 * delta)))                # 变体 A 的每题成功率
p1 = 1 / (1 + np.exp(-(a + 1.0 * delta)))                # 变体 B 的每题成功率

y0 = (rng.random((n_sim, B)) < p0).astype(float)
y1 = (rng.random((n_sim, B)) < p1).astype(float)

# 配对（RCB）：每题两种变体各做一次，估计量 = 逐题差值的均值
diffs_paired = (y1 - y0).mean(axis=1)

# 不配对（CRD）：每题随机分给一种变体，估计量 = 两组成功率之差
t = rng.random((n_sim, B)) < 0.5
ya = np.where(t, y1, y0)
nB = t.sum(axis=1)
nA = (~t).sum(axis=1)
meanB = (ya * t).sum(axis=1) / np.maximum(nB, 1)
meanA = (ya * (~t)).sum(axis=1) / np.maximum(nA, 1)
diffs_unpaired = meanB - meanA

sd_p = diffs_paired.std(ddof=1)
sd_u = diffs_unpaired.std(ddof=1)
sd_theory = np.sqrt((p0 * (1 - p0) + p1 * (1 - p1)).mean() / B)

print(f"题目数 B={B}, 难度组间SD={sigma_b}, 真实效应 delta={delta}（logit 尺度）")
print(f"平均成功率：变体A={p0.mean():.3f}, 变体B={p1.mean():.3f}")
print()
print(f"不配对（CRD）估计量：均值={diffs_unpaired.mean():+.4f}, SD={sd_u:.4f}")
print(f"配对（RCB） 估计量：均值={diffs_paired.mean():+.4f}, SD={sd_p:.4f}")
print(f"配对理论 SD = sqrt(E[p0(1-p0)+p1(1-p1)]/B) = {sd_theory:.4f}")
print(f"方差比（不配对/配对）= {sd_u**2 / sd_p**2:.1f} 倍")
print(f"SD 比（不配对/配对）= {sd_u / sd_p:.1f} 倍")

题目数 B=24, 难度组间SD=1.5, 真实效应 delta=0.5（logit 尺度）
平均成功率：变体A=0.538, 变体B=0.625

不配对（CRD）估计量：均值=+0.0867, SD=0.2047
配对（RCB） 估计量：均值=+0.0869, SD=0.1187
配对理论 SD = sqrt(E[p0(1-p0)+p1(1-p1)]/B) = 0.1199
方差比（不配对/配对）= 3.0 倍
SD 比（不配对/配对）= 1.7 倍
